In [ ]:
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


class Config:
    def __init__(self):
        self.root_dir = "/kaggle/working/dataset/"
        self.region = "left_eye"

        self.num_random_frames_per_video = 2
        self.random_seed = 42

        self.padding_ratio = 0.35
        self.aligned_face_size = 320

        self.eye_width_scale = 1.30
        self.eye_height_scale = 1.15
        self.eye_horizontal_bias = 0.05
        self.eye_vertical_bias = -0.02

        self.smoothing_alpha = 0.70

        self.model_path = "/content/face_landmarker.task"
        self.num_faces = 1
        self.min_face_detection_confidence = 0.5
        self.min_face_presence_confidence = 0.5
        self.min_tracking_confidence = 0.5


class MediaPipeTasksDetector:
    def __init__(self, model_path, num_faces=1,
                 min_face_detection_confidence=0.5,
                 min_face_presence_confidence=0.5,
                 min_tracking_confidence=0.5):
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            num_faces=num_faces,
            min_face_detection_confidence=min_face_detection_confidence,
            min_face_presence_confidence=min_face_presence_confidence,
            min_tracking_confidence=min_tracking_confidence,
            output_face_blendshapes=False,
            output_facial_transformation_matrixes=False,
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)

        self.left_eye_indices = [33, 133, 160, 159, 158, 157, 173, 246, 161, 144, 145, 153, 154, 155]
        self.right_eye_indices = [362, 263, 387, 386, 385, 384, 398, 466, 388, 373, 374, 380, 381, 382]
        self.mouth_indices = [61, 291, 13, 14, 78, 308]

    def close(self):
        self.detector.close()

    def detect(self, frame_bgr):
        h, w = frame_bgr.shape[:2]

        if len(frame_bgr.shape) == 2:
            rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2RGB)
        else:
            rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = self.detector.detect(mp_image)

        if not result.face_landmarks:
            return None

        face_landmarks = result.face_landmarks[0]

        def point(idx):
            lm = face_landmarks[idx]
            return float(lm.x * w), float(lm.y * h)

        left_eye_points = [point(i) for i in self.left_eye_indices]
        right_eye_points = [point(i) for i in self.right_eye_indices]
        mouth_points = [point(i) for i in self.mouth_indices]

        left_eye_center = (
            float(np.mean([p[0] for p in left_eye_points])),
            float(np.mean([p[1] for p in left_eye_points])),
        )
        right_eye_center = (
            float(np.mean([p[0] for p in right_eye_points])),
            float(np.mean([p[1] for p in right_eye_points])),
        )
        mouth_center = (
            float(np.mean([p[0] for p in mouth_points])),
            float(np.mean([p[1] for p in mouth_points])),
        )

        xs = [p[0] for p in left_eye_points + right_eye_points + mouth_points]
        ys = [p[1] for p in left_eye_points + right_eye_points + mouth_points]

        return {
            "facial_area": [min(xs), min(ys), max(xs), max(ys)],
            "landmarks": {
                "left_eye": left_eye_center,
                "right_eye": right_eye_center,
                "mouth_center": mouth_center,
                "left_eye_points": left_eye_points,
                "right_eye_points": right_eye_points,
            },
        }


def crop_with_reflect_padding(img, x1, y1, x2, y2):
    h, w = img.shape[:2]

    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - w)
    pad_bottom = max(0, y2 - h)

    if pad_left or pad_top or pad_right or pad_bottom:
        img = cv2.copyMakeBorder(
            img,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            borderType=cv2.BORDER_REFLECT
        )

    x1 += pad_left
    x2 += pad_left
    y1 += pad_top
    y2 += pad_top

    return img[y1:y2, x1:x2]


def align_face_3pts(frame_bgr, landmarks, out_size):
    left_eye = landmarks["left_eye"]
    right_eye = landmarks["right_eye"]
    mouth_center = landmarks["mouth_center"]

    src = np.array([
        [float(left_eye[0]), float(left_eye[1])],
        [float(right_eye[0]), float(right_eye[1])],
        [float(mouth_center[0]), float(mouth_center[1])],
    ], dtype=np.float32)

    dst = np.array([
        [0.33 * out_size, 0.40 * out_size],
        [0.67 * out_size, 0.40 * out_size],
        [0.50 * out_size, 0.72 * out_size],
    ], dtype=np.float32)

    M = cv2.getAffineTransform(src, dst)
    aligned = cv2.warpAffine(
        frame_bgr,
        M,
        (out_size, out_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT
    )
    return aligned, M


def transform_point(M, point_xy):
    px, py = point_xy
    out_x = (M[0, 0] * px) + (M[0, 1] * py) + M[0, 2]
    out_y = (M[1, 0] * px) + (M[1, 1] * py) + M[1, 2]
    return float(out_x), float(out_y)


def smooth_scalar(prev, curr, alpha):
    if prev is None:
        return float(curr)
    return float(alpha * prev + (1.0 - alpha) * curr)


def smooth_point(prev, curr, alpha):
    if prev is None:
        return float(curr[0]), float(curr[1])
    return (
        float(alpha * prev[0] + (1.0 - alpha) * curr[0]),
        float(alpha * prev[1] + (1.0 - alpha) * curr[1]),
    )


def smooth_box(prev_box, curr_box, alpha):
    if prev_box is None:
        return tuple(float(v) for v in curr_box)
    return (
        float(alpha * prev_box[0] + (1.0 - alpha) * curr_box[0]),
        float(alpha * prev_box[1] + (1.0 - alpha) * curr_box[1]),
        float(alpha * prev_box[2] + (1.0 - alpha) * curr_box[2]),
        float(alpha * prev_box[3] + (1.0 - alpha) * curr_box[3]),
    )


def estimate_eye_box_from_points(eye_points, padding_ratio, width_scale, height_scale):
    xs = [p[0] for p in eye_points]
    ys = [p[1] for p in eye_points]

    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)

    eye_w = max(x_max - x_min, 1.0)
    eye_h = max(y_max - y_min, 1.0)

    pad_scale = 1.0 + padding_ratio
    eye_w = max(int(round(eye_w * pad_scale * width_scale)), 1)
    eye_h = max(int(round(eye_h * pad_scale * height_scale)), 1)

    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0

    return center_x, center_y, eye_w, eye_h


class EyePreviewStrategy:
    def __init__(self, cfg):
        self.cfg = cfg
        self.region = cfg.region
        self.padding_ratio = cfg.padding_ratio
        self.aligned_face_size = cfg.aligned_face_size
        self.eye_width_scale = cfg.eye_width_scale
        self.eye_height_scale = cfg.eye_height_scale
        self.eye_horizontal_bias = cfg.eye_horizontal_bias
        self.eye_vertical_bias = cfg.eye_vertical_bias
        self.smoothing_alpha = cfg.smoothing_alpha

        self.prev_eye_center = None
        self.prev_eye_w = None
        self.prev_eye_h = None
        self.prev_eye_box = None

    def extract(self, frame, face):
        if face is None or "landmarks" not in face:
            return None

        landmarks = face["landmarks"]
        aligned_bgr, M = align_face_3pts(frame, landmarks, out_size=self.aligned_face_size)

        if self.region == "left_eye":
            eye_points_raw = landmarks["left_eye_points"]
        else:
            eye_points_raw = landmarks["right_eye_points"]

        eye_points_aligned = [transform_point(M, p) for p in eye_points_raw]

        eye_center_x, eye_center_y, eye_w, eye_h = estimate_eye_box_from_points(
            eye_points=eye_points_aligned,
            padding_ratio=self.padding_ratio,
            width_scale=self.eye_width_scale,
            height_scale=self.eye_height_scale,
        )

        eye_center_aligned = (eye_center_x, eye_center_y)
        eye_center_aligned = smooth_point(self.prev_eye_center, eye_center_aligned, self.smoothing_alpha)
        eye_w = smooth_scalar(self.prev_eye_w, float(eye_w), self.smoothing_alpha)
        eye_h = smooth_scalar(self.prev_eye_h, float(eye_h), self.smoothing_alpha)

        cx, cy = eye_center_aligned
        eye_w_i = max(int(round(eye_w)), 1)
        eye_h_i = max(int(round(eye_h)), 1)

        x_shift = self.eye_horizontal_bias * eye_w_i
        if self.region == "left_eye":
            x_shift = -x_shift
        y_shift = self.eye_vertical_bias * eye_h_i

        x1 = cx - (eye_w_i / 2.0) + x_shift
        y1 = cy - (eye_h_i / 2.0) + y_shift
        x2 = x1 + eye_w_i
        y2 = y1 + eye_h_i

        x1, y1, x2, y2 = smooth_box(self.prev_eye_box, (x1, y1, x2, y2), self.smoothing_alpha)

        gray_aligned = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2GRAY)
        crop = crop_with_reflect_padding(
            gray_aligned,
            int(round(x1)),
            int(round(y1)),
            int(round(x2)),
            int(round(y2))
        )

        self.prev_eye_center = eye_center_aligned
        self.prev_eye_w = eye_w
        self.prev_eye_h = eye_h
        self.prev_eye_box = (x1, y1, x2, y2)

        return {
            "crop": crop,
            "aligned_bgr": aligned_bgr,
            "eye_points_aligned": eye_points_aligned,
            "box": (int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))),
        }


def collect_train_videos(root_dir):
    items = []

    for person in sorted(os.listdir(root_dir)):
        person_path = os.path.join(root_dir, person)
        if not os.path.isdir(person_path):
            continue

        for condition in sorted(os.listdir(person_path)):
            condition_path = os.path.join(person_path, condition)
            if not os.path.isdir(condition_path):
                continue

            for file_name in sorted(os.listdir(condition_path)):
                if not file_name.lower().endswith(".avi"):
                    continue

                video_path = os.path.join(condition_path, file_name)
                video_name = os.path.splitext(file_name)[0]

                items.append({
                    "person": person,
                    "condition": condition,
                    "video_name": video_name,
                    "video_path": video_path
                })

    return items


def sample_random_frames_from_video(video_path, num_frames, rng):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    sample_count = min(num_frames, total_frames)
    frame_indices = sorted(rng.sample(range(total_frames), sample_count))

    frames = []
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            frames.append((frame_idx, frame))

    cap.release()
    return frames


def draw_original_frame(frame, face, region):
    vis = frame.copy()
    landmarks = face["landmarks"]

    if region == "left_eye":
        eye_points = landmarks["left_eye_points"]
    else:
        eye_points = landmarks["right_eye_points"]

    for x, y in eye_points:
        cv2.circle(vis, (int(round(x)), int(round(y))), 1, (0, 255, 0), -1)

    lx, ly = landmarks["left_eye"]
    rx, ry = landmarks["right_eye"]
    mx, my = landmarks["mouth_center"]

    cv2.circle(vis, (int(round(lx)), int(round(ly))), 3, (255, 0, 0), -1)
    cv2.circle(vis, (int(round(rx)), int(round(ry))), 3, (255, 0, 0), -1)
    cv2.circle(vis, (int(round(mx)), int(round(my))), 3, (0, 0, 255), -1)

    return vis


def draw_aligned_frame(aligned_bgr, eye_points_aligned, box):
    vis = aligned_bgr.copy()

    for x, y in eye_points_aligned:
        cv2.circle(vis, (int(round(x)), int(round(y))), 1, (0, 255, 0), -1)

    x1, y1, x2, y2 = box
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 2)

    return vis


def preview_all_train_eye_videos(cfg):
    rng = random.Random(cfg.random_seed)
    detector = MediaPipeTasksDetector(
        model_path=cfg.model_path,
        num_faces=cfg.num_faces,
        min_face_detection_confidence=cfg.min_face_detection_confidence,
        min_face_presence_confidence=cfg.min_face_presence_confidence,
        min_tracking_confidence=cfg.min_tracking_confidence,
    )

    try:
        videos = collect_train_videos(cfg.root_dir)
        if not videos:
            raise RuntimeError("No .avi videos found in train folder structure")

        results = []

        for item in videos:
            strategy = EyePreviewStrategy(cfg)

            sampled_frames = sample_random_frames_from_video(
                item["video_path"],
                cfg.num_random_frames_per_video,
                rng
            )

            for frame_idx, frame in sampled_frames:
                face = detector.detect(frame)

                if face is None:
                    results.append({
                        "title": f"{item['person']} | {item['condition']} | {item['video_name']} | frame {frame_idx}",
                        "original_vis": frame,
                        "aligned_vis": None,
                        "crop": None,
                        "status": "not_detected"
                    })
                    continue

                preview = strategy.extract(frame, face)

                if preview is None:
                    results.append({
                        "title": f"{item['person']} | {item['condition']} | {item['video_name']} | frame {frame_idx}",
                        "original_vis": draw_original_frame(frame, face, cfg.region),
                        "aligned_vis": None,
                        "crop": None,
                        "status": "failed"
                    })
                    continue

                original_vis = draw_original_frame(frame, face, cfg.region)
                aligned_vis = draw_aligned_frame(
                    preview["aligned_bgr"],
                    preview["eye_points_aligned"],
                    preview["box"]
                )

                results.append({
                    "title": f"{item['person']} | {item['condition']} | {item['video_name']} | frame {frame_idx}",
                    "original_vis": original_vis,
                    "aligned_vis": aligned_vis,
                    "crop": preview["crop"],
                    "status": "ok"
                })

        if not results:
            print("No preview results")
            return

        rows = len(results)
        plt.figure(figsize=(15, 4 * rows))

        plot_idx = 1
        for item in results:
            plt.subplot(rows, 3, plot_idx)
            plt.imshow(cv2.cvtColor(item["original_vis"], cv2.COLOR_BGR2RGB))
            plt.title(item["title"] + "\noriginal")
            plt.axis("off")
            plot_idx += 1

            plt.subplot(rows, 3, plot_idx)
            if item["aligned_vis"] is not None:
                plt.imshow(cv2.cvtColor(item["aligned_vis"], cv2.COLOR_BGR2RGB))
                plt.title("aligned face + eye box")
            else:
                plt.imshow(np.full((120, 120, 3), 128, dtype=np.uint8))
                plt.title(item["status"])
            plt.axis("off")
            plot_idx += 1

            plt.subplot(rows, 3, plot_idx)
            if item["crop"] is not None:
                plt.imshow(item["crop"], cmap="gray")
                plt.title("eye crop")
            else:
                plt.imshow(np.full((40, 80), 128, dtype=np.uint8), cmap="gray")
                plt.title("no eye crop")
            plt.axis("off")
            plot_idx += 1

        plt.tight_layout()
        plt.show()

    finally:
        detector.close()


if __name__ == "__main__":
    cfg = Config()
    cfg.root_dir = "/kaggle/working/dataset/"
    cfg.region = "left_eye"
    cfg.num_random_frames_per_video = 2
    cfg.model_path = "/content/face_landmarker.task"
    preview_all_train_eye_videos(cfg)